# Multimodal RAG with LangChain: PDF (Text + Table + Image) using CLIP + GPT-4V

## Architecture Overview

```
PDF Input
   ├── Text Extraction → Text Chunks → Text Embeddings (OpenAI)
   ├── Table Extraction → HTML/Markdown → Table Embeddings (OpenAI)
   └── Image Extraction → CLIP Embeddings → Image Embeddings
              ↓
     Unified Vector Store (ChromaDB / Qdrant)
              ↓
     Query → CLIP + Text Embeddings → Retrieve (text + table + images)
              ↓
     GPT-4o (Multimodal) → Final Answer
```

---

## Step 1: Install Dependencies

```bash
pip install langchain langchain-openai langchain-community
pip install unstructured[pdf] pdfminer.six pymupdf
pip install chromadb openai pillow
pip install transformers torch torchvision
pip install "unstructured[all-docs]"
```

---

## Step 2: Extract Text, Tables, and Images from PDF

```python
import fitz  # PyMuPDF
import base64
from io import BytesIO
from PIL import Image
from unstructured.partition.pdf import partition_pdf

def extract_pdf_elements(pdf_path: str):
    """Extract text, tables, and images from PDF"""
    
    # Use unstructured to extract text and tables
    elements = partition_pdf(
        filename=pdf_path,
        extract_images_in_pdf=True,
        infer_table_structure=True,
        chunking_strategy="by_title",
        max_characters=4000,
        new_after_n_chars=3800,
        combine_text_under_n_chars=2000,
        image_output_dir_path="./extracted_images/"
    )
    
    texts = []
    tables = []
    
    for element in elements:
        if "Table" in str(type(element)):
            tables.append(str(element))
        elif "CompositeElement" in str(type(element)):
            texts.append(str(element))
    
    return texts, tables

def extract_images_from_pdf(pdf_path: str):
    """Extract images using PyMuPDF"""
    doc = fitz.open(pdf_path)
    images = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        image_list = page.get_images(full=True)
        
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]
            
            # Convert to PIL Image
            pil_image = Image.open(BytesIO(image_bytes)).convert("RGB")
            
            images.append({
                "image": pil_image,
                "page": page_num + 1,
                "index": img_index,
                "bytes": image_bytes,
                "ext": image_ext
            })
    
    return images
```

---

## Step 3: Generate CLIP Embeddings for Images

```python
import torch
import numpy as np
from transformers import CLIPProcessor, CLIPModel
from langchain.embeddings.base import Embeddings
from typing import List

class CLIPImageEmbeddings:
    """CLIP-based embeddings for images"""
    
    def __init__(self, model_name="openai/clip-vit-base-patch32"):
        self.model = CLIPModel.from_pretrained(model_name)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
    
    def embed_image(self, image: Image.Image) -> List[float]:
        """Generate CLIP embedding for a single image"""
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        return image_features.cpu().numpy()[0].tolist()
    
    def embed_text_for_image_search(self, text: str) -> List[float]:
        """Generate CLIP text embedding to search images"""
        inputs = self.processor(text=[text], return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            text_features = self.model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features.cpu().numpy()[0].tolist()
    
    def embed_images_batch(self, images: List[Image.Image]) -> List[List[float]]:
        """Batch embed multiple images"""
        inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        return image_features.cpu().numpy().tolist()
```

---

## Step 4: Build Separate Vector Stores

```python
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
import uuid

# ---- Text & Table Vector Store (OpenAI Embeddings) ----
def build_text_vectorstore(texts: list, tables: list):
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    docs = []
    # Add text chunks
    for i, text in enumerate(texts):
        docs.append(Document(
            page_content=text,
            metadata={"type": "text", "chunk_id": i}
        ))
    
    # Add tables
    for i, table in enumerate(tables):
        docs.append(Document(
            page_content=table,
            metadata={"type": "table", "table_id": i}
        ))
    
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embedding_model,
        collection_name="text_table_store"
    )
    return vectorstore


# ---- Image Vector Store (CLIP Embeddings + ChromaDB) ----
def build_image_vectorstore(images: list, clip_embedder: CLIPImageEmbeddings):
    client = chromadb.Client()
    collection = client.create_collection(
        name="image_store",
        metadata={"hnsw:space": "cosine"}
    )
    
    image_store = []
    embeddings = []
    ids = []
    metadatas = []
    
    for img_data in images:
        img_id = str(uuid.uuid4())
        embedding = clip_embedder.embed_image(img_data["image"])
        
        # Store base64 image for later retrieval
        buffered = BytesIO()
        img_data["image"].save(buffered, format="PNG")
        b64_image = base64.b64encode(buffered.getvalue()).decode()
        
        ids.append(img_id)
        embeddings.append(embedding)
        metadatas.append({
            "page": img_data["page"],
            "b64_image": b64_image,   # store image inline
            "type": "image"
        })
    
    if embeddings:
        collection.add(
            ids=ids,
            embeddings=embeddings,
            metadatas=metadatas,
            documents=["image"] * len(ids)  # placeholder
        )
    
    return collection, client
```

---

## Step 5: Unified Retriever

```python
from langchain.schema.retriever import BaseRetriever
from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from typing import List

class MultimodalRetriever(BaseRetriever):
    """Retrieves text, tables, AND images for a given query"""
    
    text_vectorstore: object
    image_collection: object
    clip_embedder: object
    k_text: int = 3
    k_images: int = 2
    
    class Config:
        arbitrary_types_allowed = True
    
    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        
        docs = []
        
        # 1. Retrieve text and tables
        text_docs = self.text_vectorstore.similarity_search(query, k=self.k_text)
        docs.extend(text_docs)
        
        # 2. Retrieve images using CLIP text embedding
        query_embedding = self.clip_embedder.embed_text_for_image_search(query)
        image_results = self.image_collection.query(
            query_embeddings=[query_embedding],
            n_results=self.k_images,
            include=["metadatas", "distances"]
        )
        
        # Convert image results to Documents
        if image_results and image_results["metadatas"]:
            for meta in image_results["metadatas"][0]:
                docs.append(Document(
                    page_content="[IMAGE]",
                    metadata={
                        "type": "image",
                        "page": meta.get("page"),
                        "b64_image": meta.get("b64_image")
                    }
                ))
        
        return docs
```

---

## Step 6: GPT-4o Multimodal Chain

```python
from langchain_openai import ChatOpenAI
from langchain.schema.messages import HumanMessage, SystemMessage

def build_multimodal_prompt(query: str, retrieved_docs: List[Document]):
    """Build a multimodal message with text + images for GPT-4o"""
    
    content = []
    
    # System context
    text_context = ""
    image_b64_list = []
    
    for doc in retrieved_docs:
        if doc.metadata.get("type") == "image":
            image_b64_list.append(doc.metadata["b64_image"])
        else:
            doc_type = doc.metadata.get("type", "text")
            text_context += f"\n[{doc_type.upper()}]\n{doc.page_content}\n"
    
    # Add text context
    content.append({
        "type": "text",
        "text": f"""You are an expert assistant. Answer the user's question using the provided context.

Context (Text + Tables):
{text_context}

Question: {query}

Answer based on ALL provided context including images:"""
    })
    
    # Add retrieved images
    for b64_img in image_b64_list:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{b64_img}",
                "detail": "high"
            }
        })
    
    return [HumanMessage(content=content)]


def create_multimodal_rag_chain(retriever):
    llm = ChatOpenAI(model="gpt-4o", max_tokens=1500)
    
    def rag_chain(query: str):
        # Retrieve relevant docs (text + tables + images)
        retrieved_docs = retriever.get_relevant_documents(query)
        
        # Build multimodal prompt
        messages = build_multimodal_prompt(query, retrieved_docs)
        
        # Get response from GPT-4o
        response = llm.invoke(messages)
        return {
            "answer": response.content,
            "retrieved_docs": retrieved_docs
        }
    
    return rag_chain
```

---

## Step 7: Full Pipeline — Put It All Together

```python
import os
os.environ["OPENAI_API_KEY"] = "your-openai-key"

def build_multimodal_rag(pdf_path: str):
    print("📄 Extracting PDF elements...")
    texts, tables = extract_pdf_elements(pdf_path)
    images = extract_images_from_pdf(pdf_path)
    
    print(f"✅ Found: {len(texts)} text chunks, {len(tables)} tables, {len(images)} images")
    
    print("🔢 Building text/table vector store...")
    text_vs = build_text_vectorstore(texts, tables)
    
    print("🖼️ Building CLIP image vector store...")
    clip_embedder = CLIPImageEmbeddings()
    image_collection, chroma_client = build_image_vectorstore(images, clip_embedder)
    
    print("🔗 Creating unified retriever...")
    retriever = MultimodalRetriever(
        text_vectorstore=text_vs,
        image_collection=image_collection,
        clip_embedder=clip_embedder,
        k_text=3,
        k_images=2
    )
    
    print("🤖 Creating GPT-4o multimodal chain...")
    rag_chain = create_multimodal_rag_chain(retriever)
    
    return rag_chain

# Run it!
rag = build_multimodal_rag("your_document.pdf")

result = rag("What does the revenue chart show for Q3?")
print(result["answer"])
```

---

## Key Design Decisions

| Component | Choice | Why |
|---|---|---|
| **Text/Table Embedding** | OpenAI `text-embedding-3-small` | Best semantic accuracy for text |
| **Image Embedding** | CLIP `ViT-B/32` | Joint image-text embedding space |
| **Image Retrieval** | CLIP text → image cosine search | Query text finds relevant images |
| **Vector DB** | ChromaDB (separate collections) | Simple, local, fast |
| **LLM** | GPT-4o | Natively multimodal |
| **PDF Parser** | Unstructured + PyMuPDF | Handles complex layouts |

---

## Tips & Best Practices

- **Image captioning fallback**: If CLIP retrieval is weak, generate captions with GPT-4o first, then store captions as text embeddings alongside the image
- **Table summarization**: Summarize tables with LLM before embedding for better semantic search
- **CLIP model size**: Use `clip-vit-large-patch14` for higher accuracy at the cost of speed
- **Chunking strategy**: Use `by_title` for structured docs, `basic` for scanned PDFs
- **Reranking**: Add a cross-encoder reranker after retrieval to improve final context quality